# Notebook 03: EfficientNet-B0 Two-Stage ABMIL Training

In [1]:
import os
import json
import time
import numpy as np
import gc
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.metrics import (
    f1_score, recall_score, roc_auc_score,
    roc_curve, confusion_matrix
)
from sklearn.calibration import calibration_curve

# Environment check 
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"timm     : {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
CUDA     : True
timm     : 1.0.26
GPU      : Tesla T4


In [2]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
OUT  = Path("/kaggle/working")

# Input files from NB02
X_TRAIN_PATH        = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH        = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH  = NB02 / "bag_ids_train.npy"
CLASS_WEIGHTS_PATH  = NB02 / "class_weights.json"

print("NB02 path exists:", NB02.exists())

NB02 path exists: True


In [3]:
MODEL_NAME   = "efficientnet_b0"
PATCH_SIZE   = 224
SEED         = 42

BS_STAGE1    = 128           
EPOCHS_S1    = 15
PATIENCE_S1  = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [4]:
# import sys
# sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
# from abmil_common import (
#     build_backbone, PatchDataset, PatchClassifier,
#     AttentionPool, BagClassifier, CachedBagDataset, collate_cached,
#     extract_features, compute_all_metrics, get_normalisation_tensors, run_cv, train_stage1_fold,
# )

import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, get_normalisation_tensors, train_stage1_fold,
)

In [5]:
print("Loading arrays...")
X_train_all  = np.load(X_TRAIN_PATH)   
y_train_all  = np.load(Y_TRAIN_PATH)  
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)  

fold_ids = np.load(NB02 / "fold_ids.npy")  

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

print("\n── Train arrays ──")
print(f"X_train_all : {X_train_all.shape}  dtype={X_train_all.dtype}"
      f"  range=[{X_train_all.min():.3f}, {X_train_all.max():.3f}]")
print(f"y_train_all : {y_train_all.shape}  "
      f"benign={(y_train_all==0).sum()}  "
      f"malignant={(y_train_all==1).sum()}")
print(f"bag_ids_all : {bag_ids_all.shape}  "
      f"unique bags={len(np.unique(bag_ids_all))}")

print(f"\n── Fold structure ──")
print(f"fold_ids : {fold_ids.shape}  unique folds={sorted(set(fold_ids))}")

print("\n── Class weights ──")
print(class_weight_dict)

Loading arrays...

── Train arrays ──
X_train_all : (58820, 224, 224, 1)  dtype=float32  range=[0.000, 1.000]
y_train_all : (58820,)  benign=29225  malignant=29595
bag_ids_all : (58820,)  unique bags=1226

── Fold structure ──
fold_ids : (1226,)  unique folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

── Class weights ──
{0: 1.0063301967493585, 1: 0.9937489440783916}


In [6]:
fold_ids = np.load(NB02 / "fold_ids.npy")

# Fold 0 used to define the fixed Stage 1 train/val split (backbone stays fixed across folds)
STAGE1_FOLD = 0
all_bags = np.unique(bag_ids_all)
train_bag_indices = all_bags[fold_ids[all_bags] != STAGE1_FOLD]
val_bag_indices   = all_bags[fold_ids[all_bags] == STAGE1_FOLD]

train_mask = np.isin(bag_ids_all, train_bag_indices)
val_mask   = np.isin(bag_ids_all, val_bag_indices)

X_tr, y_tr, bag_tr    = X_train_all[train_mask], y_train_all[train_mask], bag_ids_all[train_mask]
X_val, y_val, bag_val = X_train_all[val_mask],   y_train_all[val_mask],   bag_ids_all[val_mask]

print(f"Stage 1 fixed split (fold {STAGE1_FOLD} held out): "
      f"train_bags={len(train_bag_indices)}  val_bags={len(val_bag_indices)}")

Stage 1 fixed split (fold 0 held out): train_bags=985  val_bags=241


In [7]:
#  Load the winning Stage 1 parameters found in NB02.5 
STAGE1_PARAMETER_PATH = Path("/kaggle/input/notebooks/mfjmrizvi/2-6-optuna-run-all/effnet_b0_stage1_optuna_study.json")
with open(STAGE1_PARAMETER_PATH) as f:
    parameter = json.load(f)["best_params"]

OPTIMISER_NAME = parameter["optimiser"]
LR             = parameter["lr"]
WEIGHT_DECAY   = parameter["weight_decay"]
print(f"Using parameter: {parameter}")

all_bags = np.unique(bag_ids_all)

Using parameter: {'optimiser': 'AdamW', 'lr': 1.1527987128232396e-05, 'weight_decay': 0.00757947995334801}


In [8]:
def get_fold_split(fold, all_bags, fold_ids, bag_ids_all, model_name, out_dir):
    """Returns train_idx, val_idx (integer positions into X_train_all/y_train_all),
    plus the save path for this fold's checkpoint."""
    if fold == "full":
        rng = np.random.default_rng(999)
        shuffled = rng.permutation(all_bags)
        n_val = int(0.1 * len(shuffled))
        val_bag_indices   = shuffled[:n_val]
        train_bag_indices = shuffled[n_val:]
        save_path = out_dir / f"{model_name}_stage1_full.pth"
    else:
        train_bag_indices = all_bags[fold_ids[all_bags] != fold]
        val_bag_indices   = all_bags[fold_ids[all_bags] == fold]
        save_path = out_dir / f"{model_name}_stage1_fold{fold}.pth"

    # np.where instead of boolean-mask slicing later — this returns index
    # POSITIONS, not a copy of the data itself. Keeps X_train_all as one
    # single array in RAM the whole time, instead of a fresh multi-GB
    # copy every fold (that copying is what caused your OOM).
    train_idx = np.where(np.isin(bag_ids_all, train_bag_indices))[0]
    val_idx   = np.where(np.isin(bag_ids_all, val_bag_indices))[0]

    return train_idx, val_idx, save_path

In [9]:
fold_histories = {}

for fold in [0, 1, 2, 3, 4, "full"]:
    print(f"\n{'='*60}\nTraining Stage 1: {MODEL_NAME} fold={fold}\n{'='*60}")

    train_idx, val_idx, save_path = get_fold_split(
        fold, all_bags, fold_ids, bag_ids_all, MODEL_NAME, OUT
    )
    print(f"Train patches: {len(train_idx)}  Val patches: {len(val_idx)}")

    history, best_val_loss = train_stage1_fold(
        model_name=MODEL_NAME,
        X_all=X_train_all, y_all=y_train_all,
        train_idx=train_idx, val_idx=val_idx,
        class_weight_dict=class_weight_dict,
        optimiser_name=OPTIMISER_NAME, lr=LR, weight_decay=WEIGHT_DECAY,
        save_path=save_path, device=DEVICE,
        epochs=EPOCHS_S1, patience=PATIENCE_S1
    )

    fold_histories[str(fold)] = {
        "best_val_loss": best_val_loss,
        "parameters": parameter,
        "epochs_budget": EPOCHS_S1,
        "patience": PATIENCE_S1,
        "train_patches": len(train_idx),
        "val_patches": len(val_idx),
        "history": history,
    }

    gc.collect(); torch.cuda.empty_cache()


Training Stage 1: efficientnet_b0 fold=0
Train patches: 47252  Val patches: 11568


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  Ep 01/15  train=0.6516  val=0.6572  auc=0.6473
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 02/15  train=0.6028  val=0.6165  auc=0.6955
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 03/15  train=0.5760  val=0.6119  auc=0.6973
    ✓ Saved: efficientnet_b0_stage1_fold0.pth
  Ep 04/15  train=0.5534  val=0.6236  auc=0.6932
  Ep 05/15  train=0.5314  val=0.6220  auc=0.6966
  Ep 06/15  train=0.5092  val=0.6318  auc=0.6935
  Ep 07/15  train=0.4841  val=0.6373  auc=0.7018
  Ep 08/15  train=0.4572  val=0.6802  auc=0.6785
  Early stopping at epoch 8

Training Stage 1: efficientnet_b0 fold=1
Train patches: 47355  Val patches: 11465
  Ep 01/15  train=0.6516  val=0.6650  auc=0.5978
    ✓ Saved: efficientnet_b0_stage1_fold1.pth
  Ep 02/15  train=0.5930  val=0.6619  auc=0.6163
    ✓ Saved: efficientnet_b0_stage1_fold1.pth
  Ep 03/15  train=0.5628  val=0.6683  auc=0.6229
  Ep 04/15  train=0.5386  val=0.6840  auc=0.6188
  Ep 05/15  train=0.5177  val=0.6896  auc=0.6176
  Ep 06/15  train=0.49

In [10]:
with open(OUT / f"{MODEL_NAME}_stage1_all_folds_summary.json", "w") as f:
    json.dump(fold_histories, f, indent=2)

print("\nAll 6 Stage 1 backbones trained (5 CV folds + 1 production).")


All 6 Stage 1 backbones trained (5 CV folds + 1 production).


In [11]:
# backbone = build_backbone(MODEL_NAME, pretrained=True)
# backbone = backbone.to(DEVICE)

# # Confirms feature dimension 
# with torch.no_grad():
#     _dummy = torch.zeros(2, 3, PATCH_SIZE, PATCH_SIZE,
#                          device=DEVICE)
#     _out   = backbone(_dummy)
#     print(f"Backbone output shape : {_out.shape}")
#     print(f"Expected FEAT_DIM     : {FEAT_DIM}")
#     assert _out.shape[1] == FEAT_DIM, (
#         f"Feature dim mismatch: got {_out.shape[1]}, "
#         f"expected {FEAT_DIM}"
#     )
# del _dummy, _out
# print("Backbone verified.")

In [12]:
# patch_model = PatchClassifier(backbone, FEAT_DIM).to(DEVICE)

# pos_weight_val = class_weight_dict[1] / class_weight_dict[0]
# criterion_s1   = nn.BCEWithLogitsLoss(
#     pos_weight=torch.tensor(pos_weight_val, device=DEVICE)
# )
# print(f"pos_weight for BCEWithLogitsLoss: {pos_weight_val:.4f}")

# optimiser_s1  = optim.Adam(patch_model.parameters(), lr=LR_STAGE1)
# scheduler_s1  = optim.lr_scheduler.ReduceLROnPlateau(
#     optimiser_s1, mode='min', factor=0.5,
#     patience=3
# )

# train_patch_ds = PatchDataset(X_tr,  y_tr)
# val_patch_ds   = PatchDataset(X_val, y_val)

# train_patch_dl = DataLoader(train_patch_ds, batch_size=BS_STAGE1,
#                              shuffle=True,  num_workers=2,
#                              pin_memory=True)
# val_patch_dl   = DataLoader(val_patch_ds,   batch_size=BS_STAGE1,
#                              shuffle=False, num_workers=2,
#                              pin_memory=True)

# print(f"Train batches : {len(train_patch_dl)}")
# print(f"Val   batches : {len(val_patch_dl)}")

In [13]:
# scaler = torch.amp.GradScaler('cuda')

# s1_history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
# best_val_loss_s1    = float("inf")
# patience_counter_s1 = 0
# t0 = time.time()

# for epoch in range(1, EPOCHS_S1 + 1):
#     patch_model.train()
#     running_loss = 0.0
#     for patches, labels in train_patch_dl:
#         # 1. Push raw single-channel patches to GPU memory immediately
#         patches, labels = patches.to(DEVICE), labels.to(DEVICE) 
        
#         # 2. Parallelised channel expansion and normalisation on GPU cores
#         patches = patches.repeat(1, 3, 1, 1)              
#         patches = (patches - _mean_gpu) / _std_gpu             
        
#         optimiser_s1.zero_grad()
        
#         # 3. Forward pass wrapped in AMP autocast to optimise VRAM
#         with torch.amp.autocast('cuda'):
#             logits = patch_model(patches)
#             loss = criterion_s1(logits, labels)
            
#         scaler.scale(loss).backward()
#         scaler.step(optimiser_s1)
#         scaler.update()
        
#         running_loss += loss.item() * len(labels)
#     train_loss = running_loss / len(train_patch_dl.dataset)

#     # Validation Phase
#     patch_model.eval()
#     val_loss, all_probs, all_labels = 0.0, [], []
#     with torch.no_grad():
#         for patches, labels in val_patch_dl:
#             patches, labels = patches.to(DEVICE), labels.to(DEVICE)

#             patches = patches.repeat(1, 3, 1, 1)
#             patches = (patches - _mean_gpu) / _std_gpu
            
#             with torch.amp.autocast('cuda'):
#                 logits   = patch_model(patches)
#                 loss_val = criterion_s1(logits, labels)
                
#             val_loss += loss_val.item() * len(labels)
#             all_probs.extend(torch.sigmoid(logits).cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
            
#     val_loss  /= len(val_patch_dl.dataset)
#     all_probs  = np.array(all_probs)
#     all_labels = np.array(all_labels)
#     val_preds  = (all_probs >= 0.5).astype(int)
#     val_auc    = roc_auc_score(all_labels, all_probs)
#     val_f1     = f1_score(all_labels, val_preds, zero_division=0)

#     s1_history["train_loss"].append(train_loss)
#     s1_history["val_loss"].append(val_loss)
#     s1_history["val_auc"].append(val_auc)
#     s1_history["val_f1"].append(val_f1)
#     scheduler_s1.step(val_loss)

#     print(f"Ep {epoch:02d}/{EPOCHS_S1}  "
#           f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
#           f"val_auc={val_auc:.4f}  val_f1={val_f1:.4f}")

#     if val_loss < best_val_loss_s1:
#         best_val_loss_s1    = val_loss
#         patience_counter_s1 = 0
#         torch.save(patch_model.state_dict(), S1_WEIGHTS)
#         print("  ✓ Saved best Stage 1 weights")
#     else:
#         patience_counter_s1 += 1
#         if patience_counter_s1 >= PATIENCE_S1:
#             print(f"  Early stopping at epoch {epoch}")
#             break

# print(f"\nStage 1 complete : {(time.time()-t0)/60:.1f} min")

In [14]:
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# axes[0].plot(s1_history["train_loss"], label="Train loss")
# axes[0].plot(s1_history["val_loss"],   label="Val loss")
# axes[0].set_title("Stage 1: Loss")
# axes[0].legend()
# axes[0].set_xlabel("Epoch")

# axes[1].plot(s1_history["val_auc"], label="Val AUC", color="orange")
# axes[1].set_title("Stage 1: Val AUC")
# axes[1].legend()
# axes[1].set_xlabel("Epoch")

# plt.tight_layout()
# plt.savefig(OUT / "stage1_training_curves.png", dpi=150)
# plt.show()
# print("Saved stage1_training_curves.png")

In [15]:
# # Stage 1 patch-level diagnostic metrics
# patch_model.load_state_dict(
#     torch.load(S1_WEIGHTS, map_location=DEVICE))
# patch_model.eval()

# all_probs, all_labels = [], []
# with torch.no_grad():
#     for patches, labels in val_patch_dl:
#         patches = patches.to(DEVICE)
        
#         patches = patches.repeat(1, 3, 1, 1)
#         patches = (patches - _mean_gpu) / _std_gpu
        
#         with torch.amp.autocast('cuda'):
#             all_probs.extend(
#                 torch.sigmoid(patch_model(patches)).cpu().numpy())
#         all_labels.extend(labels.numpy())

# all_probs  = np.array(all_probs)
# all_labels = np.array(all_labels)
# preds      = (all_probs >= 0.5).astype(int)
# tn, fp, fn, tp = confusion_matrix(all_labels, preds).ravel()

# s1_metrics = {
#     "patch_auc"        : float(roc_auc_score(all_labels, all_probs)),
#     "patch_f1"         : float(f1_score(all_labels, preds, zero_division=0)),
#     "patch_sensitivity": float(recall_score(all_labels, preds, zero_division=0)),
#     "patch_specificity": float(tn / (tn + fp)),
# }
# print("Stage 1 Patch-Level Val Metrics")
# for k, v in s1_metrics.items():
#     print(f"  {k:25s}: {v:.4f}")

In [16]:
# # Frozen Feature Extraction
# feature_extractor = patch_model.backbone
# feature_extractor.eval()
# for p in feature_extractor.parameters():
#     p.requires_grad_(False)

# print("Extracting features using uniform NB02 patch cache...")
# feats_tr = extract_features(X_tr, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# feats_val = extract_features(X_val, feature_extractor, _mean_gpu, _std_gpu, DEVICE)
# # feats_test = extract_features(X_test_all, feature_extractor, _mean_gpu, _std_gpu, DEVICE)

# print(f"  Train cache: {feats_tr.shape}")
# print(f"  Val cache:   {feats_val.shape}")
# # print(f"  Test cache:  {feats_test.shape}")

In [17]:
# import gc

# # X_tr / X_val were only needed for Stage 1 training — CV runs on cached
# # features, not raw patches, so free the raw arrays before extracting for all bags
# del X_tr, X_val
# gc.collect()
# torch.cuda.empty_cache()

In [18]:
# print(f"Running 5-fold CV for Stage 2 ({MODEL_NAME})...")
# feats_all = extract_features(X_train_all, feature_extractor, _mean_gpu, _std_gpu, DEVICE)

# # Once features are cached, the raw pixel array is no longer needed either —
# # feats_all is (53787, FEAT_DIM) ≈ hundreds of MB, vs X_train_all's ~10.8 GB
# del X_train_all
# gc.collect()

In [19]:
# cv_results, cv_summary = run_cv(
#     feats_all, y_train_all, bag_ids_all, fold_ids, feat_dim=FEAT_DIM,
#     attn_dim=ATTN_DIM, dropout=0.25, gated=True, tag=MODEL_NAME,
#     device=DEVICE
# )
# print(cv_summary)

# with open(OUT / f"{MODEL_NAME}_cv_results.json", "w") as f:
#     json.dump({
#         "model": MODEL_NAME, "feat_dim": FEAT_DIM,
#         "fold_results": cv_results,
#         "cv_mean": cv_summary.loc["mean"].to_dict(),
#         "cv_std": cv_summary.loc["std"].to_dict(),
#     }, f, indent=2)
# print("Saved:", f"{MODEL_NAME}_cv_results.json")

In [20]:
# from sklearn.metrics import ConfusionMatrixDisplay

# fig, axes = plt.subplots(1, 5, figsize=(20, 4))
# for i, r in enumerate(cv_results):
#     cm = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
#     disp = ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"])
#     disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
#     axes[i].set_title(f"Fold {i}\nMCC={r['mcc']:.3f}")
# plt.suptitle(f"{MODEL_NAME} — per-fold confusion matrices (val, calibrated threshold)")
# plt.tight_layout()
# plt.savefig(OUT / f"{MODEL_NAME}_confusion_matrices.png", dpi=150)
# plt.show()